In [1]:
import os
os.chdir("../../web_backend/")

In [2]:
from tqdm import tqdm
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

from data.data import CollectionAccessor, ImageHandler, EmbeddingSpaceAccessor

from search import Search, GraphSearcher, TextEmbeddingSearcher, EmbeddingSearcher

In [3]:
from app import init_DMG, search_collection

In [4]:
def init_DMG():
    DMG_DIR = "./data/DMG"
    image_folder = DMG_DIR+"/images/"
    image_handler = ImageHandler("DMG", image_folder=image_folder, keep_prefix=False)

    time_stamp, pub_file, priv_file = CollectionAccessor.get_latest_dump(DMG_DIR+"/dumps")
    print(time_stamp)

    dmg_meta = dict(name="Design Museum Gent (public & private)", id_="DMG_"+time_stamp,
                creation_timestamp=time_stamp, language="nl")
    df = CollectionAccessor.get_DMG(pub_path=pub_file, #get_latest("./data/dumps", contains="public"),
                                     priv_path=priv_file, #get_latest("./data/dumps", contains="private"),
                                     rights_path=DMG_DIR+"/rights.csv",
                                     image_handler=image_handler,
                                     **dmg_meta)

    kg_searcher = GraphSearcher(df)


    sem_embs = EmbeddingSpaceAccessor.load(DMG_DIR+"/generated_data/distiluse-base-multilingual-cased-v2",
                                       loadXD=None)
    concept_search = TextEmbeddingSearcher(sem_embs, name="concept-searcher")


    sem_embs = EmbeddingSpaceAccessor.load(DMG_DIR+"/generated_data/distiluse-base-multilingual-cased-v2",
                                       loadXD=32)
    sem_searcher = EmbeddingSearcher(sem_embs, name="semantic-searcher")
    
    viz_embs = EmbeddingSpaceAccessor.load(DMG_DIR+"/generated_data/vitmae", loadXD=32)
    viz_searcher = EmbeddingSearcher(viz_embs, name="visual-searcher")

    s = Search([kg_searcher, sem_searcher, viz_searcher])
    return df, s, concept_search, sem_embs, sem_searcher

df, s, cs, sem_embs, sem_searcher = init_DMG()

2026-03-28


[GraphSearcher]: building graph...: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21030/21030 [00:02<00:00, 10284.15it/s]


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

---

In [21]:
no_text = df[df.coll.get_texts().str.len() < 5].coll.get_texts()

scores = sem_searcher(no_text.iloc[:1]).sort_values()

In [22]:
df.loc[scores.index[-10:]].coll.get_texts()

object_number
2015-0013_07-12    Koffiekop van het servies 'Sonja'\nPieter Stoc...
1987-0213_3-3                                                     AR
0086_0-2           portet van een vrouw\ngeheel= kader + portret ...
2001-0019_2-4      Stapelbare stoel '.03' in de oorspronkelijke k...
2000-0053_08-10    Space Saver Super Oval\ntwee onderdelen: doos ...
2009-0124          Ge Vielt... O Vorst! , muziek: Martony, E., te...
2010-0037_07-97                              Koffiekop van porselein
1024_2-5                                    Bord met rivierlandschap
4069                                            Kelkdoek, geborduurd
1393                                                            blad
dtype: str

---

In [ ]:
df.sample(1).coll.get_texts().iloc[0]

In [ ]:
cs("shadow").sort_values().iloc[-10:]

In [ ]:
df.loc[["2013-0037_2-5"]].coll.get_texts().iloc[0]

In [ ]:
df[df.coll.get_texts().str.lower().str.contains("schaduw")].coll.get_texts().tolist()

In [ ]:
cs("lelies").sort_values().iloc[-10:]

In [ ]:
df.loc["1987-0609"]

---

In [ ]:
r = df.sample(1)
r.coll.get_texts().iloc[0]

In [ ]:
scores = sem_searcher(r).sort_values()
lens = df.loc[scores.index].coll.get_texts().str.len()
lens.name = "length"
sns.jointplot(x=lens, y=scores)

In [ ]:
lens.value_counts().sort_index()

In [ ]:
df.loc[scores.index][lens < 10].coll.get_texts()

In [ ]:
df.loc["1987-0616_05-13"]

---

In [26]:
list(df.coll.get_texts().sample(4))

["Boterschaaltje 200 van het servies 'Seagull'\nBoterschaaltje met modelnummer 200 behoort tot het servies Seagull met gouden randjes. De serviesonderdelen zijn klassiek van vorm, maar het decor sluit aan bij een nieuw thema dat de art nouveau aanboorde, met name de natuur. Op een dégradé blauwe achtergrond zweeft een zeemeeuw door het luchtruim, zeepaardjes fungeren als handvatten van schalen en kommen. Het model van het servies - met schubbenmotief in reliëf - dateert uit de jaren 1880 en is van Frederik August Hallin. Het decor met de zeemeeuw is van de hand van Fanny Garde en dateert van 1895. Vanaf de jaren 1950 was het servies in één op tien Deense huishoudens te vinden, waardoor het werd bestempeld als het 'nationale Deense servies'. Er bestaat ook een versie zonder gouden randje.",
 'Stoelsport',
 'Drawer Canister 0.175 l low\nDeze stapelbare opbergdoos heeft een doorzichtig deksel om de inhoud te tonen bij gebruik in schuifladen. Ze is met een druk van de hand te openen en is 

In [27]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("tomaarsen/static-similarity-mrl-multilingual-v1") #"sentence-transformers/all-MiniLM-L6-v2")


# The sentences to encode
sentences = ["Boterschaaltje 200 van het servies 'Seagull'\nBoterschaaltje met modelnummer 200 behoort tot het servies Seagull met gouden randjes. De serviesonderdelen zijn klassiek van vorm, maar het decor sluit aan bij een nieuw thema dat de art nouveau aanboorde, met name de natuur. Op een dégradé blauwe achtergrond zweeft een zeemeeuw door het luchtruim, zeepaardjes fungeren als handvatten van schalen en kommen. Het model van het servies - met schubbenmotief in reliëf - dateert uit de jaren 1880 en is van Frederik August Hallin. Het decor met de zeemeeuw is van de hand van Fanny Garde en dateert van 1895. Vanaf de jaren 1950 was het servies in één op tien Deense huishoudens te vinden, waardoor het werd bestempeld als het 'nationale Deense servies'. Er bestaat ook een versie zonder gouden randje.",
 'Stoelsport',
 'Drawer Canister 0.175 l low\nDeze stapelbare opbergdoos heeft een doorzichtig deksel om de inhoud te tonen bij gebruik in schuifladen. Ze is met een druk van de hand te openen en is gemaakt uit polypropyleen, een lichte, harde en goedkope kunststof met goede isolerende eigenschappen. Tupperware, opgericht in 1946 door Earl Tupper, werd beroemd met de massaproductie van kunststof huishoudproducten, in het bijzonder voor het bereiden, serveren en bewaren van voedsel. In 1961 opende in België de eerste fabriek buiten de Verenigde Staten. Die kreeg in 1966 een ontwerpafdeling voor de Europese, Afrikaanse en Midden-Oosterse markt. Daar werkte Bob Daenen van 1966 tot 2004 als hoofddesigner en vanaf 2002 als Vice President Innovation Worldwide. Met dit ontwerp wonnen hij en de Deense Erik Herlow Design Studio een Highest Design Quality Award op de Reddot Design Awards 1996.',
 'Ontwerp voor "MEUBLE A DOSSIERS"\nontwerp voor meubels']

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)
print(embeddings.shape)
# [3, 384]

# 3. Calculate the embedding similarities
similarities = model.similarity(embeddings, embeddings)
print(similarities)
# tensor([[1.0000, 0.6660, 0.1046],
#         [0.6660, 1.0000, 0.1411],
#         [0.1046, 0.1411, 1.0000]])

(4, 1024)
tensor([[1.0000, 0.1439, 0.2527, 0.0922],
        [0.1439, 1.0000, 0.0936, 0.0718],
        [0.2527, 0.0936, 1.0000, 0.3542],
        [0.0922, 0.0718, 0.3542, 1.0000]])


In [ ]:
batches = np.array_split(df.coll.get_texts(), len(df)//10)

embs = np.zeros((len(df), 384))

j = 0
for i, b in enumerate(tqdm(batches)):
    embeddings = model.encode(b)

    embs[j:(j+embeddings.shape[0]), :] = embeddings
    j += embeddings.shape[0]

In [ ]:
minilm = pd.DataFrame(embs, index=df.index)

In [ ]:
r = df.sample(100).index

In [ ]:
sims1 = model.similarity(minilm.loc[r].values, minilm.loc[r].values).numpy()
sims2 = model.similarity(sem_searcher.space.loc[r].values, sem_searcher.space.loc[r].values).numpy()

In [ ]:
sims1

In [ ]:
sims2

In [ ]:
print(df.loc[r].coll.get_texts().iloc[0])


In [ ]:
df.loc[r[sims1[0].argsort()]].coll.get_texts()

In [ ]:
plt.plot(
    df.loc[r].coll.get_texts().str.len(),
    sims1[0], ".")

In [ ]:
for r1, r2 in zip(sims1, sims2):
    plt.plot(r1, r2, ".")

---

## model testing

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "DTAI-KULeuven/robbert-2022-dutch-base" #"jegormeister/bert-base-dutch-cased-snli"

tokenizer = AutoTokenizer.from_pretrained(model_name)#
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto")#,
    # torch_dtype=torch.bfloat16,
#)

input_text = "Wie is de koning van Spanje?"
input_ids = tokenizer(input_text, return_tensors="pt")

outputs = model.generate(**input_ids, max_new_tokens=32)
print(tokenizer.decode(outputs[0]))